In [1]:
!pip install safetensors torch


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [5]:
import os
import torch
from safetensors import safe_open
from safetensors.torch import save_file, load_file

def run_safetensors_demo():
    filename = "safetensors_test.safetensors"
    
    # ---------------------------------------------------------
    # Step 1: Define a Mock Model Tensor Dictionary
    # ---------------------------------------------------------
    print("--- Step 1: Simulating model weights in memory ---")
    # Imagine these are weights from an LLM attention block or linear layer
    tensors = {
        "embedding.weights": torch.randn(1024, 512, dtype=torch.float32),
        "attention.query.weight": torch.randn(512, 512, dtype=torch.float16),
        "attention.key.weight": torch.randn(512, 512, dtype=torch.float16),
        "bias_vector": torch.zeros(512, dtype=torch.float32)
    }
    
    for name, tensor in tensors.items():
        print(f"Prepared tensor '{name}' -> Shape: {list(tensor.shape)}, Dtype: {tensor.dtype}")

    # ---------------------------------------------------------
    # Step 2: Save the Tensors Securely
    # ---------------------------------------------------------
    print(f"\n--- Step 2: Saving weights to '{filename}' ---")
    # You can pass arbitrary string metadata directly into the header
    metadata = {
        "author": "Kin Yung",
        "format": "safetensors",
        "description": "Demo secure inference checkpoint"
    }
    
    save_file(tensors, filename, metadata=metadata)
    print(f"Success! Saved to disk. File size: {os.path.getsize(filename) / 1024:.2f} KB")

    # ---------------------------------------------------------
    # Step 3: Auditing file metadata (No code execution risk)
    # ---------------------------------------------------------
    print("\n--- Step 3: Auditing file metadata (No code execution risk) ---")
    with safe_open(filename, framework="pt", device="cpu") as f:
        # Pull the absolute header metadata block
        file_metadata = f.metadata()
        print(f"Embedded Metadata: {file_metadata}")
        
        # Look at individual layer topologies instantly
        print("Tensor Layout Table of Contents:")
        for key in f.keys():
            slice_info = f.get_slice(key)
            # Convert the list to a string first using str() to allow left-alignment formatting
            shape_str = str(slice_info.get_shape())
            print(f"  - Layer: {key:<25} | Shape: {shape_str:<12} | Dtype: {slice_info.get_dtype()}")

    # ---------------------------------------------------------
    # Step 4: Full High-Performance Load
    # ---------------------------------------------------------
    print("\n--- Step 4: Loading entire file back via Zero-Copy mmap ---")
    # SafeTensors uses OS memory mapping to bind files instantly
    loaded_tensors = load_file(filename)
    
    # Verification assert check
    assert torch.equal(tensors["bias_vector"], loaded_tensors["bias_vector"])
    print("Verification pass: Loaded data perfectly matches original inputs.")
    
    # Clean up the demo artifact file
    # if os.path.exists(filename):
        # os.remove(filename)
        # print(f"\nCleaned up local file '{filename}'.")

if __name__ == "__main__":
    run_safetensors_demo()

--- Step 1: Simulating model weights in memory ---
Prepared tensor 'embedding.weights' -> Shape: [1024, 512], Dtype: torch.float32
Prepared tensor 'attention.query.weight' -> Shape: [512, 512], Dtype: torch.float16
Prepared tensor 'attention.key.weight' -> Shape: [512, 512], Dtype: torch.float16
Prepared tensor 'bias_vector' -> Shape: [512], Dtype: torch.float32

--- Step 2: Saving weights to 'safetensors_test.safetensors' ---
Success! Saved to disk. File size: 3074.45 KB

--- Step 3: Auditing file metadata (No code execution risk) ---
Embedded Metadata: {'description': 'Demo secure inference checkpoint', 'author': 'Kin Yung', 'format': 'safetensors'}
Tensor Layout Table of Contents:
  - Layer: attention.key.weight      | Shape: [512, 512]   | Dtype: F16
  - Layer: attention.query.weight    | Shape: [512, 512]   | Dtype: F16
  - Layer: bias_vector               | Shape: [512]        | Dtype: F32
  - Layer: embedding.weights         | Shape: [1024, 512]  | Dtype: F32

--- Step 4: Loadin